> 在线讲义：[第 34 章 · 信息几何：Fisher 信息、KL 的局部二次近似、自然梯度、TRPO / PPO 的信任域](https://wikieden.github.io/math-and-eval-stats/lessons/34-information-geometry.html) · 本书《数学与实验统计》

# 第 34 章 · 信息几何：Fisher 信息、KL 的局部二次近似、自然梯度、TRPO / PPO 的信任域

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/34-information-geometry.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 Fisher | score² 均值 | `fisher_bernoulli(0.9)` | 11.1111 vs 11.0849 |
| 第二步 KL 近似 | ½Fd² | `kl_quadratic(0.5, 0.1)` | 0.020411 vs 0.02 |
| 第三步 自然梯度 | 三种更新 | `fit_coin("gd_logit")` | 2552 步 |
| 第四步 TRPO | 信任域步 | `trpo_step(np.array([1.0, 1.0]))` | [0.0408, 0.0816] |
| 第五步 softmax | diag(p) − ppᵀ | `softmax_fisher(p)` | 特征值含 0 |

例子：伯努利分布的 Fisher 信息和自然梯度（拟合硬币正面概率）；高斯策略 N(μ, σ²) 在 (μ, log σ) 坐标下的 Fisher 矩阵和
TRPO 的信任域步；softmax 策略的 Fisher 矩阵（奇异）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `fisher_bernoulli`**：伯努利(p) 的 Fisher 信息：解析 1/(p(1−p))，和 score² 的样本均值比。score = x/p − (1−x)/(1−p)。

In [ ]:
def fisher_bernoulli(p: float, n: int = 400000, seed: int = 0) -> tuple[float, float]:
    """伯努利(p) 的 Fisher 信息：解析 1/(p(1−p))，和 score² 的样本均值比。score = x/p − (1−x)/(1−p)。"""
    x = (np.random.default_rng(seed).random(n) < p).astype(float)
    s = x / p - (1 - x) / (1 - p)
    return 1 / (p * (1 - p)), float((s ** 2).mean())

**函数 `kl_bern`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def kl_bern(p: float, q: float) -> float:
    return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))

**函数 `kl_quadratic`**：KL(p ‖ p + d) 和二阶近似 ½ F d²。

In [ ]:
def kl_quadratic(p: float, d: float) -> tuple[float, float]:
    """KL(p ‖ p + d) 和二阶近似 ½ F d²。"""
    return kl_bern(p, p + d), 0.5 * d * d / (p * (1 - p))

**函数 `fit_coin`**：最大化平均对数似然 ℓ(p) = p̂ log p + (1 − p̂) log(1 − p)，看要几步到 |p − p̂| < tol。

In [ ]:
def fit_coin(method: str, p_hat: float = 0.9, p0: float = 0.5, lr: float = 0.05, tol: float = 1e-6, max_it: int = 100000) -> int:
    """最大化平均对数似然 ℓ(p) = p̂ log p + (1 − p̂) log(1 − p)，看要几步到 |p − p̂| < tol。
    gd_p：在 p 上做梯度上升（∇ = (p̂ − p)/(p(1−p))，lr = 0.05）；gd_logit：在 logit η 上做（∇ = p̂ − p，lr = 0.05）；
    natural：自然梯度 F⁻¹∇，在 p 坐标里 = p̂ − p，步长 1。"""
    p = p0
    for it in range(1, max_it + 1):
        if method == "gd_p":
            p = np.clip(p + lr * (p_hat - p) / (p * (1 - p)), 1e-6, 1 - 1e-6)
        elif method == "gd_logit":
            eta = np.log(p / (1 - p)) + lr * (p_hat - p)
            p = 1 / (1 + np.exp(-eta))
        else:
            p = p + 1.0 * (p_hat - p)
        if abs(p - p_hat) < tol:
            return it
    return max_it

**函数 `kl_gauss`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def kl_gauss(m1, s1, m2, s2) -> float:
    return float(np.log(s2 / s1) + (s1 ** 2 + (m1 - m2) ** 2) / (2 * s2 ** 2) - 0.5)

**函数 `trpo_step`**：高斯策略参数 θ = (μ, log σ)，Fisher = diag(1/σ², 2)。信任域 max gᵀδ s.t. ½δᵀFδ ≤ ε 的解：

In [ ]:
def trpo_step(g: np.ndarray, sigma: float = 0.5, eps: float = 0.01) -> tuple[np.ndarray, float]:
    """高斯策略参数 θ = (μ, log σ)，Fisher = diag(1/σ², 2)。信任域 max gᵀδ s.t. ½δᵀFδ ≤ ε 的解：
    δ = √(2ε / (gᵀF⁻¹g))·F⁻¹g。返回 (δ, 走完这一步后的真实 KL)。"""
    f = np.diag([1 / sigma ** 2, 2.0])
    nat = np.linalg.solve(f, g)
    d = np.sqrt(2 * eps / (g @ nat)) * nat
    return d, kl_gauss(0.0, sigma, d[0], sigma * np.exp(d[1]))

**函数 `softmax_fisher`**：softmax 策略对 logits 的 Fisher = diag(p) − ppᵀ；所有 logits 同加一个常数分布不变，所以有一个零特征值。

In [ ]:
def softmax_fisher(p: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """softmax 策略对 logits 的 Fisher = diag(p) − ppᵀ；所有 logits 同加一个常数分布不变，所以有一个零特征值。"""
    f = np.diag(p) - np.outer(p, p)
    return f, np.linalg.eigvalsh(f)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 Fisher 信息：score 的方差 = 对数似然曲率的期望")
for p in (0.5, 0.9):
    a, m = fisher_bernoulli(p)
    print(f"  伯努利 p = {p}：解析 1/(p(1−p)) = {a:.4f}，score² 的样本均值 {m:.4f}")

In [ ]:
print("\n第二步 KL 的局部二阶近似：KL(p ‖ p + d) ≈ ½ F d²")
for p, d in ((0.5, 0.1), (0.5, 0.01), (0.9, 0.05)):
    e, q = kl_quadratic(p, d)
    print(f"  p = {p}、d = {d}：KL = {e:.6f}，½Fd² = {q:.6f}")

In [ ]:
print("\n第三步 自然梯度：按 KL 量步长，换参数化也不变")
for m in ("gd_p", "gd_logit", "natural"):
    print(f"  {m}：从 0.5 拟合到 p̂ = 0.9 用了 {fit_coin(m)} 步")

In [ ]:
print("\n第四步 TRPO：在 KL 球里走最陡的方向")
d, kl = trpo_step(np.array([1.0, 1.0]))
print(f"  σ = 0.5，梯度 g = [1, 1]，ε = 0.01：δ = {np.round(d, 4).tolist()}，二阶近似 KL = 0.01，真实 KL = {kl:.5f}")
plain = np.array([1.0, 1.0]) * np.sqrt(2 * 0.01 / (np.array([1.0, 1.0]) @ np.diag([4.0, 2.0]) @ np.array([1.0, 1.0])))
print(f"  对比：沿普通梯度方向走到同样的 KL，δ = {np.round(plain, 4).tolist()}（μ 方向走得更少）")

In [ ]:
print("\n第五步 softmax 策略的 Fisher 是奇异的")
f, ev = softmax_fisher(np.array([0.7, 0.2, 0.1]))
print(f"  p = [0.7, 0.2, 0.1]：特征值 {(np.round(ev, 4) + 0.0).tolist()}（有一个 0：logits 整体平移不改变策略）")

## 3. 代码里才看得到的

1. **p 坐标的梯度上升要截断到 (0, 1)**：步长大时 p 可能越界，代码用 `np.clip`；自然梯度和 logit 坐标天然不会越界。
2. **TRPO 的"真实 KL"用的是高斯 KL 闭式**：log σ 参数要先 exp 回 σ 再代公式，所以真实 KL（0.00915）和二次近似（0.01）不完全相等。
3. **零特征值打印成 −0.0**：浮点误差让它略小于 0，代码加 0.0 把 −0.0 变成 0.0（第 22 章的浮点细节）。

## 4. 自己改一改

1. 把 `fit_coin` 的目标改成 p̂ = 0.99，三种方法的步数怎么变？
2. 把 TRPO 例子的 σ 改成 2，看自然梯度在 μ 方向的步长怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> Fisher 是 score 的方差、似然的曲率；
> KL 局部是二次型，F 是分布空间的尺；
> 自然梯度 F 逆乘梯度，换参数也不变；
> TRPO 线性目标加二次 KL，共轭梯度求方向；
> PPO 截比率是便宜替身，softmax 的 F 要加阻尼。